# 20 · Reusable functions, DataFrame.transform and classes

**Scenario:** Build modular transformations without collecting records.

**Source:** Databricks-provided `samples.nyctaxi.trips` only. File formats, nested structures, replays and histories in this lab are derived from that asset. No hand-created record dataset, third-party download, pandas or generated replacement data is used.

**Prerequisites:** Run notebook 00 first. Each notebook is independent; run its cells in order. Mode: **dataframe**. 

**Learning-size sample:** `max_rows` defaults to 2,000. Sorting/limiting caps result size, not the underlying scan cost. Assertions use relationships instead of hardcoded dataset counts because sample assets may change.


## Step 1 — Load and inspect the Databricks asset

Read the built-in sample and normalize column names. A stable content fingerprint helps with replay examples; it is not a guaranteed real-world trip key. The original `samples` asset is always read-only.


In [ ]:
from pyspark.sql import functions as F, types as T, Window
dbutils.widgets.text('max_rows','2000')
MAX_ROWS=int(dbutils.widgets.get('max_rows'))
assert 100<=MAX_ROWS<=20000, 'Use 100–20000 rows for these bounded learning labs.'
SOURCE_ASSET='samples.nyctaxi.trips'
try:
    asset=(spark.read.table(SOURCE_ASSET)
        .select('tpep_pickup_datetime','tpep_dropoff_datetime','trip_distance',
                'fare_amount','pickup_zip','dropoff_zip')
        .orderBy('tpep_pickup_datetime','tpep_dropoff_datetime','pickup_zip',
                 'dropoff_zip','trip_distance','fare_amount')
        .limit(MAX_ROWS))
    assert asset.limit(1).count()==1, 'Databricks sample asset is empty.'
except Exception as exc:
    raise RuntimeError('This course requires the Databricks-provided samples.nyctaxi.trips asset. '
        'Check Unity Catalog/sample access on the attached compute. No synthetic or external fallback is used.') from exc
df=(asset.select(F.col('tpep_pickup_datetime').cast('timestamp').alias('pickup_ts'),
    F.col('tpep_dropoff_datetime').cast('timestamp').alias('dropoff_ts'),
    F.col('trip_distance').cast('double').alias('distance'),
    F.col('fare_amount').cast('double').alias('fare'),
    F.col('pickup_zip').cast('string').alias('pickup_zip'),
    F.col('dropoff_zip').cast('string').alias('dropoff_zip')))
df=df.withColumn('trip_id',F.sha2(F.to_json(F.struct(*[F.col(c) for c in df.columns]),
    options={'ignoreNullFields':'false'}),256))
df=df.withColumn('pickup_date',F.to_date('pickup_ts'))
df=df.withColumn('duration_sec',F.unix_timestamp('dropoff_ts')-F.unix_timestamp('pickup_ts'))
print('Source:',SOURCE_ASSET,'sample limit:',MAX_ROWS)
# trip_id is a content fingerprint, NOT an official source/business trip identifier.
# Identical source rows intentionally have identical fingerprints.
df.show(5,truncate=False)


## Step 2 — Separate transformation from execution

A pure transformation returns a DataFrame and leaves writes/actions to the caller. It is easier to test and reuse.


In [ ]:
def valid_trips(frame):
    return frame.filter((F.col('distance')>0)&(F.col('fare')>=0)&(F.col('duration_sec')>=0))
def add_metrics(frame):
    return frame.withColumn('fare_per_mile',F.col('fare')/F.col('distance'))
result=df.transform(valid_trips).transform(add_metrics)
assert result.filter('fare_per_mile IS NULL').count()==0
result.show(5)


**Expected check:** The composed transformation produces valid per-mile measures.


## Step 3 — Use configuration in a class

A class is useful when related transformations share configuration. Do not hide network writes inside a generic property or transform.


In [ ]:
from dataclasses import dataclass
@dataclass(frozen=True)
class TripPipeline:
    minimum_distance:float=0.0
    def transform(self,frame):
        return frame.filter(F.col('distance')>self.minimum_distance).transform(valid_trips).transform(add_metrics)
pipeline=TripPipeline(minimum_distance=1.0)
out=pipeline.transform(df)
assert out.filter('distance <= 1').count()==0
print('Pipeline rows:',out.count())


**Expected check:** All output distances exceed the configured threshold.


## Practice

Move the class into a workspace Python module and import it in two notebooks.

## Solution guide

Use an absolute workspace/module path or package installation supported by your compute. Keep transforms independent of dbutils; resolve widgets/secrets only in the entry notebook.

## Production considerations

Keep source lineage and stable business keys. Measure on representative volumes, review nulls and time semantics, and make retries idempotent. Sample-data checks demonstrate mechanics; they do not establish production performance or exactly-once external writes.
